# Week 46, Day 2: BST insert and delete

Companion notebook. The interactive figures, X-ray views, quizzes and auto-marked exercises are on the [lesson page](https://peter-adepoju.github.io/python-atlas/days/w046d2.html). Use this notebook to keep your own copy of the code.

**How to use it:** run each cell with Shift+Enter. Change the code and run it again: that's how you learn.

In [ ]:
# Run this cell once. It lets you check exercise answers inside this notebook.
# Workflow: run your exercise cell, then immediately run the "check" cell below it.
import io, builtins

def check(tests_src):
    """Run an exercise's tests against the cell you ran just before this one."""
    ip = get_ipython()
    code = ip.user_ns["In"][-2]
    def run(feed=(), echo=False):
        out = io.StringIO(); q = [str(x) for x in feed]; real_in, real_print = builtins.input, builtins.print
        def fake_input(p=""):
            if not q:
                raise AssertionError("The program asked for more input than the test gave it.")
            v = q.pop(0); out.write(str(p) + (v + "\n" if echo else "")); return v
        def fake_print(*a, **k):
            k.setdefault("file", out); real_print(*a, **k)
        builtins.input, builtins.print = fake_input, fake_print
        try:
            exec(code, {"__name__": "__main__"})
        finally:
            builtins.input, builtins.print = real_in, real_print
        return out.getvalue()
    norm = lambda t: "\n".join(l.rstrip() for l in str(t).rstrip().splitlines())
    def expect_output(want, inputs=(), echo=False):
        got = run(inputs, echo)
        assert norm(got) == norm(want), f"expected {norm(want)!r} but got {norm(got)!r}"
    import re
    uses = lambda *names: all(re.search(r"\b" + re.escape(n) + r"\b", code) for n in names)
    def run_with(subs, feed=(), echo=False):
        nonlocal code
        saved = code
        for old, new in subs.items():
            code = code.replace(old, new, 1)
        try:
            return run(feed, echo)
        finally:
            code = saved
    env = dict(ip.user_ns, run=run, norm=norm, source=code, expect_output=expect_output, uses=uses, run_with=run_with)
    try:
        env["output"] = run()
    except AssertionError:
        env["output"] = ""
    exec(tests_src, env)
    tests = [(k, v) for k, v in env.items() if k.startswith("test_") and callable(v)]
    ok = 0
    for name, fn in tests:
        label = (fn.__doc__ or name[5:].replace("_", " ")).strip()
        try:
            fn(); ok += 1; print("✓", label)
        except AssertionError as e:
            print("✗", label + ":", e or "not quite yet")
    print(f"{ok} of {len(tests)} checks passed.")


**Predict before you run:** what will this print?



In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def insert(node, val):
    if node is None:
        return TreeNode(val)
    if val < node.val:
        node.left = insert(node.left, val)
    elif val > node.val:
        node.right = insert(node.right, val)
    return node

def preorder(node):
    return [] if node is None else [node.val] + preorder(node.left) + preorder(node.right)

root = None
for v in [5, 2, 8, 2, 6, 1, 9]:
    root = insert(root, v)
print(preorder(root))

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def insert(root, val):
    """Insert val; return (root, added)."""
    new = TreeNode(val)
    if root is None:
        return new, True
    parent, node = None, root
    while node is not None:
        if val == node.val:
            return root, False
        parent, node = node, (node.left if val < node.val else node.right)
    if val < parent.val:
        parent.left = new
    else:
        parent.right = new
    return root, True

def inorder(node):
    return [] if node is None else inorder(node.left) + [node.val] + inorder(node.right)

root = None
for v in [40, 20, 60, 10, 30, 50, 70, 30]:
    root, added = insert(root, v)
    if not added:
        print("already there:", v)
print(inorder(root))

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def successor(root, val):
    """The smallest value greater than val, or None."""
    best, node = None, root
    while node is not None:
        if val < node.val:
            best, node = node.val, node.left     # a candidate; maybe a smaller one to the left
        else:
            node = node.right
    return best

T = TreeNode
root = T(50, T(30, T(20), T(40, T(35))), T(70, T(60), T(80, None, T(90))))
print([successor(root, v) for v in [20, 40, 50, 55, 90]])

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def delete(node, val):
    if node is None:
        return None                               # not found: nothing changes
    if val < node.val:
        node.left = delete(node.left, val)
    elif val > node.val:
        node.right = delete(node.right, val)
    else:
        if node.left is None:                     # cases 1 and 2
            return node.right
        if node.right is None:
            return node.left
        succ = node.right                          # case 3
        while succ.left is not None:
            succ = succ.left
        node.val = succ.val
        node.right = delete(node.right, succ.val)
    return node

def inorder(node):
    return [] if node is None else inorder(node.left) + [node.val] + inorder(node.right)

T = TreeNode
root = T(50, T(30), T(70, T(60, None, T(65)), T(80)))
for v in [50, 30, 99, 70]:
    root = delete(root, v)
    print(f"after deleting {v}: root {root.val}, {inorder(root)}")

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

class BST:
    def __init__(self, values=()):
        self._root, self._size = None, 0
        for v in values:
            self.add(v)

    def __len__(self):
        return self._size

    def __contains__(self, val):
        node = self._root
        while node is not None and node.val != val:
            node = node.left if val < node.val else node.right
        return node is not None

    def add(self, val):
        if val in self:
            return
        self._root = self._insert(self._root, val)
        self._size += 1

    def _insert(self, node, val):
        if node is None:
            return TreeNode(val)
        if val < node.val:
            node.left = self._insert(node.left, val)
        else:
            node.right = self._insert(node.right, val)
        return node

    def min(self):
        node = self._root
        while node.left is not None:
            node = node.left
        return node.val

s = BST([5, 3, 8, 3, 1])
print(len(s), 3 in s, 4 in s, s.min())

In [ ]:
class Node:
    def __init__(self, key, value):
        self.key, self.value, self.left, self.right = key, value, None, None

class SortedMap:
    def __init__(self):
        self._root, self._size = None, 0

    def __setitem__(self, key, value):
        if self._root is None:
            self._root, self._size = Node(key, value), 1
            return
        node = self._root
        while True:
            if key == node.key:
                node.value = value                       # update, size unchanged
                return
            side = "left" if key < node.key else "right"
            child = getattr(node, side)
            if child is None:
                setattr(node, side, Node(key, value))
                self._size += 1
                return
            node = child

    def __getitem__(self, key):
        node = self._root
        while node is not None:
            if key == node.key:
                return node.value
            node = node.left if key < node.key else node.right
        raise KeyError(key)

    def __len__(self):
        return self._size

prices = SortedMap()
for k, v in [("pear", 40), ("apple", 30), ("fig", 90), ("apple", 35)]:
    prices[k] = v
print(len(prices), prices["apple"], prices["fig"])
try:
    prices["kiwi"]
except KeyError as e:
    print("KeyError", e)

> **Key idea.** Insert by searching and attaching a new leaf where the search falls off; the recursive style returns each subtree (`node.left = insert(node.left, val)`). A successor is the minimum of the right subtree, or else the last ancestor where the search went left. Delete has three cases: a leaf is removed, a node with one child is replaced by it, and a node with two children takes its successor's value, after which the successor is deleted from the right subtree. Insert, delete, successor and search are all O(h). Wrap the tree in a class that keeps a size, and store key and value pairs to get a sorted map; store counts to allow duplicates.

### Exercise: Inserting, two ways

Write `insert_loop(root, val)` (a loop; return the root) and `insert_rec(node, val)` (recursion returning the subtree). Both ignore values already present. The checks insert 20,000 increasing values with `insert_loop`, so it mustn't recurse.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def insert_loop(root, val):
    ___

def insert_rec(node, val):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef inorder(n):\n    out, stack = [], []\n    while stack or n:\n        while n:\n            stack.append(n)\n            n = n.left\n        n = stack.pop()\n        out.append(n.val)\n        n = n.right\n    return out\ndef pre(n):\n    return [] if n is None else [n.val] + pre(n.left) + pre(n.right)\ndef test_insert():\n    \"inserting\"\n    for f in (insert_loop, insert_rec):\n        root = None\n        for v in [5, 2, 8, 2, 6, 1, 9]:\n            root = f(root, v)\n        assert pre(root) == [5, 2, 1, 8, 6, 9], f.__name__\n        vals = random.Random(2).sample(range(1000), 300)\n        root = None\n        for v in vals:\n            root = f(root, v)\n        assert inorder(root) == sorted(vals)\n    root = None\n    for v in range(20_000):\n        root = insert_loop(root, v)\n    assert len(inorder(root)) == 20_000")

### Exercise: Successor and predecessor

Write `successor(root, val)` and `predecessor(root, val)`: the smallest value greater than `val` and the largest value less than `val`, or `None`. `val` need not be in the tree. One walk from the root each.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def successor(root, val):
    ___

def predecessor(root, val):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, bisect\nT = TreeNode\ndef test_succ():\n    \"next and previous\"\n    root = T(50, T(30, T(20), T(40, T(35))), T(70, T(60), T(80, None, T(90))))\n    assert [successor(root, v) for v in [20, 40, 50, 55, 90]] == [30, 50, 60, 60, None]\n    assert [predecessor(root, v) for v in [20, 35, 50, 55, 100]] == [None, 30, 40, 50, 90]\n    vals = sorted(random.Random(4).sample(range(200), 50))\n    def build(lo, hi):\n        if lo > hi:\n            return None\n        m = (lo + hi) // 2\n        return T(vals[m], build(lo, m - 1), build(m + 1, hi))\n    r = build(0, len(vals) - 1)\n    for x in range(-2, 202):\n        i = bisect.bisect_right(vals, x)\n        assert successor(r, x) == (vals[i] if i < len(vals) else None)\n        j = bisect.bisect_left(vals, x)\n        assert predecessor(r, x) == (vals[j - 1] if j else None)")

### Exercise: Deleting

Write `delete(node, val)` returning the new subtree, handling all three cases (use the successor for two children). Deleting a missing value changes nothing.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def delete(node, val):
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef ins(n, v):\n    if n is None:\n        return TreeNode(v)\n    if v < n.val:\n        n.left = ins(n.left, v)\n    elif v > n.val:\n        n.right = ins(n.right, v)\n    return n\ndef inorder(n):\n    return [] if n is None else inorder(n.left) + [n.val] + inorder(n.right)\ndef ok(n, lo=float(\"-inf\"), hi=float(\"inf\")):\n    return n is None or (lo < n.val < hi and ok(n.left, lo, n.val) and ok(n.right, n.val, hi))\ndef test_cases():\n    \"three cases\"\n    T = TreeNode\n    r = delete(T(50, T(30), T(70, T(60, None, T(65)), T(80))), 50)\n    assert r.val == 60 and r.right.left.val == 65 and inorder(r) == [30, 60, 65, 70, 80]\n    assert delete(T(1), 1) is None and inorder(delete(T(5, T(3)), 5)) == [3] and inorder(delete(T(5, T(3)), 9)) == [3, 5]\ndef test_random():\n    \"random\"\n    rng = random.Random(5)\n    for _ in range(30):\n        vals = rng.sample(range(100), 40)\n        root = None\n        for v in vals:\n            root = ins(root, v)\n        live = set(vals)\n        for v in rng.sample(range(100), 60):\n            root = delete(root, v)\n            live.discard(v)\n            assert inorder(root) == sorted(live) and ok(root)")

### Exercise: A set built on a BST

Finish the `BST` class: `add`, `discard` (no error if missing), `__contains__`, `__len__` (kept as a count, not recounted), `min()` and `max()` (`ValueError` when empty), and `__iter__` returning an `InOrder` iterator object (write the class; no `yield`).

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

class InOrder:
    ___

class BST:
    def __init__(self, values=()):
        self._root, self._size = None, 0
        for v in values:
            self.add(v)
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_set():\n    \"a sorted set\"\n    s = BST([5, 3, 8, 3, 1])\n    assert len(s) == 4 and 3 in s and 4 not in s and list(s) == [1, 3, 5, 8] and s.min() == 1 and s.max() == 8\n    s.discard(5)\n    s.discard(42)\n    assert len(s) == 3 and list(s) == [1, 3, 8] and isinstance(iter(s), InOrder)\n    e = BST()\n    assert list(e) == [] and len(e) == 0\n    for f in (e.min, e.max):\n        try:\n            f()\n            assert False\n        except ValueError:\n            pass\ndef test_random():\n    \"against a set\"\n    rng = random.Random(6)\n    s, ref = BST(), set()\n    for _ in range(3000):\n        v = rng.randrange(200)\n        if rng.random() < 0.6:\n            s.add(v)\n            ref.add(v)\n        else:\n            s.discard(v)\n            ref.discard(v)\n        assert len(s) == len(ref)\n    assert list(s) == sorted(ref)")

### Exercise: A sorted map

Write `SortedMap` with `m[key] = value` (insert or update), `m[key]` (`KeyError` if missing), `del m[key]` (`KeyError` if missing), `key in m`, `len(m)`, `keys()` (a sorted list), `items()` (a sorted list of pairs) and `floor_key(k)` (the largest key ≤ `k`, or `None`). Deleting a node with two children must move the successor's key **and** value.

In [ ]:
class Node:
    def __init__(self, key, value):
        self.key, self.value, self.left, self.right = key, value, None, None

class SortedMap:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\ndef test_map():\n    \"a sorted map\"\n    m = SortedMap()\n    for k, v in [(\"pear\", 40), (\"apple\", 30), (\"fig\", 90), (\"apple\", 35), (\"kiwi\", 20)]:\n        m[k] = v\n    assert len(m) == 4 and m[\"apple\"] == 35 and \"fig\" in m and \"plum\" not in m\n    assert m.keys() == [\"apple\", \"fig\", \"kiwi\", \"pear\"] and m.items()[0] == (\"apple\", 35)\n    assert m.floor_key(\"grape\") == \"fig\" and m.floor_key(\"a\") is None and m.floor_key(\"zz\") == \"pear\"\n    del m[\"fig\"]\n    assert m.keys() == [\"apple\", \"kiwi\", \"pear\"] and len(m) == 3\n    for f in (lambda: m[\"fig\"], lambda: m.__delitem__(\"fig\")):\n        try:\n            f()\n            assert False\n        except KeyError:\n            pass\ndef test_random():\n    \"against a dict\"\n    rng = random.Random(7)\n    m, ref = SortedMap(), {}\n    for i in range(3000):\n        k = rng.randrange(150)\n        r = rng.random()\n        if r < 0.55:\n            m[k] = i\n            ref[k] = i\n        elif k in ref:\n            del m[k]\n            del ref[k]\n        assert len(m) == len(ref)\n    assert m.items() == sorted(ref.items())")

### Exercise: Counting duplicates

Write `MultiSet`, a BST whose nodes hold a value and a `count`: `add(v)` (new node or count + 1), `remove(v)` (count − 1, deleting the node at 0; `KeyError` if absent), `count(v)`, `len()` (the total including repeats) and `to_list()` (sorted, with repeats).

In [ ]:
class Node:
    def __init__(self, val):
        self.val, self.count, self.left, self.right = val, 1, None, None

class MultiSet:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random\nfrom collections import Counter\ndef test_multiset():\n    \"counts\"\n    m = MultiSet()\n    for v in [3, 1, 3, 2, 3]:\n        m.add(v)\n    assert m.to_list() == [1, 2, 3, 3, 3] and len(m) == 5 and m.count(3) == 3 and m.count(9) == 0\n    m.remove(3)\n    m.remove(1)\n    assert m.to_list() == [2, 3, 3] and len(m) == 3\n    try:\n        m.remove(1)\n        assert False\n    except KeyError:\n        pass\n    rng = random.Random(8)\n    m, ref = MultiSet(), Counter()\n    for _ in range(2000):\n        v = rng.randrange(30)\n        if rng.random() < 0.6:\n            m.add(v)\n            ref[v] += 1\n        elif ref[v]:\n            m.remove(v)\n            ref[v] -= 1\n    assert m.to_list() == sorted(ref.elements()) and len(m) == sum(ref.values())")

### Bug hunt: The lost leaf

This recursive insert calls itself but throws away what comes back, so a new leaf is made and then dropped: only the first value survives and this prints `[5]`. Store the returned subtree so it prints `[2, 5, 8]`.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def insert(node, val):
    if node is None:
        return TreeNode(val)
    if val < node.val:
        insert(node.left, val)
    elif val > node.val:
        insert(node.right, val)
    return node

def inorder(node):
    return [] if node is None else inorder(node.left) + [node.val] + inorder(node.right)

root = None
for v in [5, 8, 2]:
    root = insert(root, v)
print(inorder(root))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"keep the result\"\n    expect_output(\"[2, 5, 8]\")")

### Bug hunt: Deleting the wrong successor

For a node with two children, this copies the successor's value up, then deletes that value from the **left** subtree, where it isn't, so the successor stays and appears twice: this prints `[30, 60, 60, 70, 80]`. Delete it from the right subtree so it prints `[30, 60, 70, 80]`.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

def delete(node, val):
    if node is None:
        return None
    if val < node.val:
        node.left = delete(node.left, val)
    elif val > node.val:
        node.right = delete(node.right, val)
    else:
        if node.left is None:
            return node.right
        if node.right is None:
            return node.left
        succ = node.right
        while succ.left is not None:
            succ = succ.left
        node.val = succ.val
        node.left = delete(node.left, succ.val)
    return node

def inorder(node):
    return [] if node is None else inorder(node.left) + [node.val] + inorder(node.right)

T = TreeNode
print(inorder(delete(T(50, T(30), T(70, T(60), T(80))), 50)))

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"from the right\"\n    expect_output(\"[30, 60, 70, 80]\")")

### Bug hunt: Counting what isn't there

This set's `discard` lowers the size even when the value isn't present, so after discarding a missing value, `len` is wrong: this prints `2 [1, 3, 5]`. Only count a removal that happened, so it prints `3 [1, 3, 5]`.

In [ ]:
class TreeNode:
    def __init__(self, val, left=None, right=None):
        self.val, self.left, self.right = val, left, right

class BST:
    def __init__(self):
        self.root, self.size = None, 0

    def contains(self, val):
        node = self.root
        while node is not None and node.val != val:
            node = node.left if val < node.val else node.right
        return node is not None

    def add(self, val):
        if not self.contains(val):
            self.root = self._ins(self.root, val)
            self.size += 1

    def _ins(self, node, val):
        if node is None:
            return TreeNode(val)
        if val < node.val:
            node.left = self._ins(node.left, val)
        else:
            node.right = self._ins(node.right, val)
        return node

    def discard(self, val):
        self.root = self._del(self.root, val)
        self.size -= 1

    def _del(self, node, val):
        if node is None:
            return None
        if val < node.val:
            node.left = self._del(node.left, val)
        elif val > node.val:
            node.right = self._del(node.right, val)
        else:
            if node.left is None:
                return node.right
            if node.right is None:
                return node.left
            succ = node.right
            while succ.left is not None:
                succ = succ.left
            node.val = succ.val
            node.right = self._del(node.right, succ.val)
        return node

    def values(self):
        def walk(n):
            return [] if n is None else walk(n.left) + [n.val] + walk(n.right)
        return walk(self.root)

s = BST()
for v in [3, 1, 5]:
    s.add(v)
s.discard(4)
print(s.size, s.values())

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_output():\n    \"only real removals\"\n    expect_output(\"3 [1, 3, 5]\")\n    s.discard(3)\n    assert s.size == 2 and s.values() == [1, 5]")

### Puzzle: Deleting a node

Arrange the lines into a recursive BST delete. One line is a trap.

(Put these lines in order on the website.)

### Exercise: Predict them



In [ ]:
# Start from the BST built by inserting 50, 30, 70, 20, 40, 60, 80, 65 in that order.

p = ___   # its pre-order traversal
q = ___   # the successor of 40
r = ___   # the root after deleting 50 (using the successor)
s = ___   # the pre-order after deleting 50
t = ___   # how many nodes have two children in the original tree

In [ ]:
# Check: run the exercise cell above first, then this one.
check("def test_values():\n    \"each\"\n    assert p == [50, 30, 20, 40, 70, 60, 65, 80] and q == 50 and r == 60, \"p, q and r\"\n    assert s == [60, 30, 20, 40, 70, 65, 80] and t == 3, \"s and t\"")

### Quiz

Take this quiz on the website: it is marked for you and feeds your daily review.

### Exercise: Challenge: ranks in O(h)

Asking "how many values are smaller than x?" or "what's the 1,000th smallest?" costs O(n) in a plain BST. Store in every node the **size of its subtree** and both become O(h). Write `RankedSet` with:

- `add(v)` and `discard(v)`, keeping every size on the path correct (only when something actually changes);
- `len(s)`, in O(1) from the root's size;
- `rank(x)`: how many values are **less than** `x` (`x` need not be present), walking one path: going right past a node adds the size of its left subtree plus 1;
- `select(k)`: the value with exactly `k` smaller values (k from 0), raising `IndexError` out of range;
- `count_between(lo, hi)`: how many values lie in `lo..hi` inclusive, using two ranks.

The checks compare against a sorted list over thousands of random operations.

In [ ]:
class Node:
    def __init__(self, val):
        self.val, self.left, self.right, self.size = val, None, None, 1

class RankedSet:
    ___

In [ ]:
# Check: run the exercise cell above first, then this one.
check("import random, bisect\ndef test_ranked():\n    \"against a sorted list\"\n    rng = random.Random(9)\n    s, ref = RankedSet(), []\n    for _ in range(4000):\n        v = rng.randrange(500)\n        i = bisect.bisect_left(ref, v)\n        present = i < len(ref) and ref[i] == v\n        if rng.random() < 0.65:\n            s.add(v)\n            if not present:\n                ref.insert(i, v)\n        else:\n            s.discard(v)\n            if present:\n                ref.pop(i)\n        assert len(s) == len(ref)\n    for x in range(-1, 502, 7):\n        assert s.rank(x) == bisect.bisect_left(ref, x)\n    for k in range(0, len(ref), 5):\n        assert s.select(k) == ref[k]\n    for lo, hi in [(0, 499), (100, 200), (250, 250), (300, 100)]:\n        want = max(0, bisect.bisect_right(ref, hi) - bisect.bisect_left(ref, lo))\n        assert s.count_between(lo, hi) == want\n    for k in (-1, len(ref)):\n        try:\n            s.select(k)\n            assert False\n        except IndexError:\n            pass\ndef test_sizes():\n    \"sizes stay right\"\n    s = RankedSet()\n    for v in [5, 3, 8, 3, 1]:\n        s.add(v)\n    s.discard(42)\n    assert len(s) == 4 and s._root.size == 4 and s.select(0) == 1 and s.rank(6) == 3")

## Warm-up

The second 2 found a node with its own value and changed nothing: this BST, like a set, keeps one copy. Notice the shape of `insert`: it **returns** the (possibly new) subtree, and every caller stores what comes back. That pattern makes deletion much easier too.

## Inserting

To insert, search for the value. If you find it, there's nothing to do (or, for a map, update the value). If the search falls off the tree, that empty spot is exactly where the value belongs: attach a new leaf there. New values are always added as **leaves**, and nothing else moves.

There are two common ways to write it:

- **A loop** that walks down keeping track of the parent, then attaches the leaf to the parent's left or right. No recursion limit to worry about.
- **Recursion that returns the subtree** (the warm-up): `node.left = insert(node.left, val)`. The `None` case makes and returns the new node, and every other level returns itself unchanged, so the assignment only really changes something at the very bottom.

The second style is short and generalises well: tomorrow's balancing code uses exactly this shape, with a rebalancing step on the way back up.

## Successor and predecessor

A value's **successor** is the next larger value in the tree (what comes after it in sorted order); its **predecessor** is the next smaller one. There are two places the successor of a node can be:

- If the node has a **right subtree**, the successor is the **minimum** of that subtree: go right once, then left as far as possible.
- Otherwise, it's the nearest **ancestor** for which the node lies in the **left** subtree. Searching from the root for the value and remembering the last node where you went left finds it.

This version is the ceiling search from yesterday with `>` instead of `>=`, and it works even for values not in the tree. The predecessor is the mirror image.

## Deleting

Find the node, then:

1. **No children** (a leaf): remove it; its parent's link becomes `None`.
2. **One child**: the child takes the node's place. Everything in the child's subtree was already on the correct side of every ancestor, so the BST property still holds.
3. **Two children**: you can't simply remove it, because two subtrees would need one slot. Instead, copy the node's **successor** (the minimum of its right subtree) into it, then delete the successor from the right subtree. The successor is bigger than everything on the left and smaller than everything else on the right, so it fits exactly; and it has no left child (it's a minimum), so deleting it is case 1 or 2.

```text
delete 50:            copy successor 60 up:       delete the old 60:
        50                    60                          60
       /  \                  /  \                        /  \
     30    70              30    70                    30    70
          /  \                  /  \                        /  \
        60    80              60    80                    65    80
          \                     \
           65                    65
```

The predecessor (the maximum of the left subtree) works just as well. The recursive, subtree-returning style handles all three cases neatly:

Deleting costs O(h), like inserting: one walk down to the node and, in case 3, one more down to its successor.

## A BST class

Passing roots in and out of functions gets clumsy. A class holds the root and a count, so `len` is O(1), and its methods hide the subtree-returning helpers:

To make the set iterable with `for v in s`, `__iter__` returns an iterator object, like Week 45 Day 3's `InOrder`; one exercise below adds it. Notice how `add` checks membership first so the size stays right; `discard` will need the same care.

## A sorted map

Store a **key and a value** in each node, order the tree by key, and you have a **sorted map**: a dictionary whose keys come out in order, with floor and ceiling and range queries thrown in. Python's `dict` keeps insertion order, not sorted order, and its hash table can't answer "the largest key below 100" without looking at every key. Many standard libraries offer a tree-based map for exactly this (Java's `TreeMap`, C++'s `std::map`); in Python, people use a third-party package or write one.

**Duplicates.** To keep repeated values (a multiset), store a **count** in each node: inserting an existing value increments it, and deleting decrements it, only removing the node when the count reaches 0.

## Practice

## Quiz

## Daily challenge

## Before you go

On paper, draw the tree from "Predict them", delete 30 (two children), then 70, and check the in-order traversal is still sorted after each. Tomorrow: keeping the tree short however the values arrive, with AVL rotations.